### Bag of Stories
**Description:** Compute Jensen-Shannon divergence in a set of stories.

**Contributor:** Florentina Armaselu  

In [6]:
# Import required packages
from pathlib import Path
import numpy as np
from scipy.spatial.distance import jensenshannon
from collections import Counter
import datetime as dt
import os
import math
import pandas as pd
import matplotlib.pyplot as plt

Define the input, output paths.

In [7]:
# Define the base directory relative to the current script location
base_dir = os.getcwd()
# Set the path to the input data folder for whole stories
input = os.path.join(base_dir, "./../data/project_gutenberg/")
# Set the path to the input data folder for segmented stories with text tiling
input_seg_tt = os.path.join(base_dir, "./../data/episode-annotated_stories/text-tiling/")
# Set the path to the output folder
output = os.path.join(base_dir, "./../results/jsd/")

Define the functions for computing Jensen-Shannon divergence in whole stories.

In [8]:
# Divide stories into segments of approximately equal length in number of tokens
def segment_story(story, seg_length=100):
    """
    Segments a story into smaller parts of approximately equal length in tokens.
    
    Args:
        story (str): The full text of the story.
        segment_length (int): The desired length of each segment in tokens.
        
    Returns:
        list: A list of segments, each containing approximately `segment_length` tokens.
    """   
    # Split the story into tokens
    tokens = story.split()
    
    # Calculate the number of segments based on the desired segment length
    seg_count = math.ceil(len(tokens) / seg_length)
    
    # Length of each segment in tokens more equally distributed
    distrib_length = math.ceil(len(tokens) / seg_count) 
    
    # Create segments of approximately equal length
    # This will ensure that the segments are more evenly distributed in terms of token count
    segments = []
    for i in range(0, len(tokens), distrib_length):
        segment_tokens = tokens[i:i + distrib_length]
        segment = ' '.join(segment_tokens)
        segments.append(segment)

    return segments

In [10]:
# Compute Jensen-Shannon divergence for two adjacent segments
def compute_adjacent_jsd(segment1, segment2):
    """
    Computes the Jensen-Shannon divergence between two segments.
    
    Args:
        segment1 (str): The first segment of text.
        segment2 (str): The second segment of text.
        
    Returns:
        float: The Jensen-Shannon divergence between the two segments.
    """
    # Tokenize the segments
    tokens1 = segment1.split()
    tokens2 = segment2.split()
    
    # Count the frequency of each token in both segments
    freq_list1 = Counter(tokens1)
    freq_list2 = Counter(tokens2)
    
    # Create a combined set of all unique tokens
    all_tokens = set(freq_list1.keys()).union(set(freq_list2.keys()))
    
    # Create probability distributions for both segments
    prob1 = np.array([freq_list1[token] / len(tokens1) if len(tokens1) > 0 else 0 for token in all_tokens])
    prob2 = np.array([freq_list2[token] / len(tokens2) if len(tokens2) > 0 else 0 for token in all_tokens])
    
    # Compute the Jensen-Shannon divergence
    jsd = jensenshannon(prob1, prob2, base=2) ** 2
    
    return jsd

In [17]:
# Compute Jensen-Shannon divergence for all adjacent segments in a story
def compute_adjacent_jsd_for_story(story):
    """
    Computes the Jensen-Shannon divergence for all adjacent segments in a story.
    
    Args:
        story (str): The full text of the story.
        
    Returns:
        list: A list of Jensen-Shannon divergence values for each pair of adjacent segments.
    """
    # Segment the story into smaller parts
    segments = segment_story(story)
    
    # Compute JSD for each pair of adjacent segments
    jsd_values = []
    for i in range(len(segments) - 1):
        print("Segment ", i, ": ", segments[i], "| Segment ", (i+1), ":", segments[i + 1])
        jsd = compute_adjacent_jsd(segments[i], segments[i + 1])
        jsd_values.append(jsd)
    
    return jsd_values

Define the functions for analysing the stories and saving the results.

In [14]:
# Process a set of stories to compute JSD values for adjacent segments and save results as CSV and PNG files
def process_adj_jsd_stories():
    """
    Processes all stories in the input directory and saves the JSD results to the output directory.
    
    Args:
        input_dir (str): The directory containing the input stories.
        output_dir (str): The directory where the results will be saved.
    """
    cnt_processed = 0

    # Read the story files sequentially from the input folder.
    for file_path in Path(input).glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as input_file:
            story = input_file.read()
    
            # Print the name of the file being processed.
            print(f'Processing file: {file_path.name}')

            # Get the title of the story
            title = story.split('\n')[0]

            # Get the body of the story, excluding the title
            body = story[story.find('\n')+1:]  
                
            # Compute JSD values for the story
            jsd_values = compute_adjacent_jsd_for_story(body)

            # Create a timestamp for the output files
            timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')

            # Save adjacent JSD table as CSV
            df = pd.DataFrame({
                'Segment': range(1, len(jsd_values) + 1),
                'JSD': jsd_values
            })

            # Create a story name from the title of the story
            story_name = title.title()
            
            # Save the DataFrame to a CSV file
            csv_path = os.path.join(output, "csv/", f"{file_path.stem}_adj_jsd_{timestamp}.csv")
            df.to_csv(csv_path, index=False)

            # Save line plot as PNG
            plt.figure(figsize=(8, 4))
            plt.plot(df['Segment'], df['JSD'], marker='o')
            plt.xlabel('Segment')
            plt.ylabel('Jensen-Shannon Divergence')
            plt.title(f'Surprisal (JSD) across Adjacent Segments: {story_name}')
            plt.grid(True)
            plt.tight_layout()
            png_path = os.path.join(output, "png/", f"{file_path.stem}_adj_jsd_{timestamp}.png")
            plt.savefig(png_path)
            plt.close()

            cnt_processed += 1

    # Print the number of files saved in the output folder.
    print(f'End processing, {cnt_processed}, files saved to {Path(output).name} folder.')

Read the stories, analyse them, and save the results.  

In [ ]:
# Call the function to process stories
process_adj_jsd_stories()